<a href="https://colab.research.google.com/github/quiet-phantom/Reviewpulse-analytics/blob/main/ReviewPulse_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install streamlit textblob wordcloud plotly -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 49.5 MB/s eta 0:00:00


In [3]:
%%writefile app.py
import streamlit as st
import pandas as pd
from textblob import TextBlob
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import plotly.express as px

st.set_page_config(page_title="ReviewPulse Analytics", page_icon="🛍️", layout="wide")
st.title("🛍️ ReviewPulse Analytics - Customer Review Insights Engine")
st.write("Upload product reviews in CSV format to analyze sentiment, distribution, and key terms.")

def get_sentiment(text):
    blob = TextBlob(str(text))
    polarity = blob.sentiment.polarity
    if polarity > 0.05:
        return "Positive", polarity
    elif polarity < -0.05:
        return "Negative", polarity
    else:
        return "Neutral", polarity

uploaded_file = st.file_uploader("Upload CSV File (must contain a 'review' or 'text' column)", type=["csv"])

if uploaded_file is not None:
    # Error Fix: CSV reading with error handling for extra commas
    try:
        df = pd.read_csv(uploaded_file, on_bad_lines='skip')
    except Exception:
        uploaded_file.seek(0)
        df = pd.read_csv(uploaded_file, engine='python', on_bad_lines='skip')

    st.subheader("📋 Raw Data Preview")
    st.dataframe(df.head())

    possible_cols = [col for col in df.columns if 'review' in col.lower() or 'text' in col.lower() or 'comment' in col.lower()]
    default_col = possible_cols[0] if possible_cols else df.columns[0]

    review_col = st.selectbox("Select Review Column:", df.columns, index=list(df.columns).index(default_col))

    if st.button("Run Sentiment Analysis"):
        with st.spinner("Analyzing customer feedback..."):
            results = df[review_col].apply(get_sentiment)
            df['Sentiment'] = [r[0] for r in results]
            df['Polarity'] = [r[1] for r in results]

            st.success("Analysis Complete!")
            st.markdown("---")

            col1, col2, col3, col4 = st.columns(4)
            total_reviews = len(df)
            pos_count = (df['Sentiment'] == 'Positive').sum()
            neg_count = (df['Sentiment'] == 'Negative').sum()
            neu_count = (df['Sentiment'] == 'Neutral').sum()

            col1.metric("Total Reviews", total_reviews)
            col2.metric("Positive 😊", f"{pos_count} ({pos_count/total_reviews*100:.1f}%)")
            col3.metric("Negative 😡", f"{neg_count} ({neg_count/total_reviews*100:.1f}%)")
            col4.metric("Neutral 😐", f"{neu_count} ({neu_count/total_reviews*100:.1f}%)")

            chart_col1, chart_col2 = st.columns(2)

            with chart_col1:
                st.subheader("📊 Sentiment Distribution")
                fig_pie = px.pie(
                    df,
                    names='Sentiment',
                    color='Sentiment',
                    color_discrete_map={'Positive':'#2ecc71', 'Negative':'#e74c3c', 'Neutral':'#f1c40f'}
                )
                st.plotly_chart(fig_pie, use_container_width=True)

            with chart_col2:
                st.subheader("☁️ Word Cloud (Customer Voice)")
                all_text = " ".join(df[review_col].astype(str))
                wordcloud = WordCloud(width=400, height=250, background_color="white").generate(all_text)
                fig_wc, ax = plt.subplots(figsize=(6, 3))
                ax.imshow(wordcloud, interpolation='bilinear')
                ax.axis('off')
                st.pyplot(fig_wc)

            st.markdown("---")
            st.subheader("🔍 Filter & Export Analyzed Data")
            selected_sentiment = st.multiselect("Filter by Sentiment:", options=["Positive", "Negative", "Neutral"], default=["Positive", "Negative", "Neutral"])

            filtered_df = df[df['Sentiment'].isin(selected_sentiment)]
            st.dataframe(filtered_df[[review_col, 'Sentiment', 'Polarity']])

            csv_data = filtered_df.to_csv(index=False).encode('utf-8')
            st.download_button(
                label="📥 Download Processed Results CSV",
                data=csv_data,
                file_name="analyzed_customer_reviews.csv",
                mime="text/csv"
            )

Overwriting app.py


In [4]:
!curl ipv4.icanhazip.com

35.189.167.71


In [5]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared

In [ ]:
!streamlit run app.py & ./cloudflared tunnel --url http://localhost:8501

2026-10-09T13:32:18Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-09T13:32:18Z INF Requesting new quick Tunnel on trycloudflare.com...


2026-10-09 13:32:19.999 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.189.167.71:8501

2026-10-09T13:32:22Z INF +-------------------------------------------------------------